In [5]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main/code
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [7]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import decentralized_train_loop, decentralized_validate_loop, decentralized_train_n_epochs
from src.data_utils import create_batched_dataloaders, create_dataloader

In [9]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns

## Parameter Setting

In [11]:
model = "umf"
val_loader_type = "oaat"
train_loader_type = "oaat"
userprop = None
n_factors = 30
sparse = False
batch_size = 10
lr = 0.012098247288774554
weight_decay = 0.051267232285266244
graph_seed = 1
n_epochs = 15

## Main

In [15]:
train_df = pd.read_csv("dataset/ml100k_train.csv")
test_df = pd.read_csv("dataset/ml100k_test.csv")
train_df.head()

,user_id,item_id,rating,rating.1
0,207,87,5,5
1,675,900,4,4
2,373,230,2,2
3,377,564,3,3
4,725,843,3,3


In [17]:
n_users = train_df['user_id'].nunique()
n_items = train_df['item_id'].nunique()
print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")

Total User: 943
Total Item: 1628


In [19]:
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)
train_data_loader = create_dataloader(
        df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=userprop
    )
val_data_loader = create_dataloader(df=val_df, dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)


In [21]:
users = {}
for i in tqdm(range(n_users)):
    # model = MF(n_users=n_users, n_items=n_items)
    user_model = UMF(n_items, n_factors = n_factors, sparse = sparse)
    # model = GeneralizedMFOneLayer(n_users=n_users, n_items=n_items)
    users[i] = User(id=i, model=user_model, optimizer=SGD(user_model.parameters(), lr=lr, weight_decay=weight_decay), model_name = model)

  0%|          | 0/943 [00:00<?, ?it/s]

In [23]:
graph = random_k_out_graph(n=n_users, k=5, seed=graph_seed)

In [ ]:
train_losses, val_losses, time_per_epoch = decentralized_train_n_epochs(
        user_models=users,
        train_loader=train_data_loader,
        val_loader=val_data_loader,
        epochs=n_epochs,
        graph=graph,
    )

  0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 4.0539 | Validation Loss: 1.8847 | Time Elapsed: 104.200816 sec

  0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 1.3994 | Validation Loss: 1.5438 | Time Elapsed: 104.802452 sec

  0%|          | 0/60000 [00:00<?, ?it/s]

Epoch 3 | Train Loss: 0.9847 | Validation Loss: 1.3881 | Time Elapsed: 104.736270 sec

  0%|          | 0/60000 [00:00<?, ?it/s]

In [ ]:
test_loss = decentralized_validate_loop(users, test_data_loader)

In [ ]:
val_losses

[5.4776874,
 1.8899466,
 1.5497322,
 1.3875123,
 1.2821261,
 1.2075137,
 1.1520663,
 1.108211,
 1.0734912,
 1.045232,
 1.0212778,
 1.0027438,
 0.9845142,
 0.97163945,
 0.960291,
 0.95081824]

In [ ]:
test_loss

0.9487044